In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("cynthiarempel/amazon-us-customer-reviews-dataset")

print("Path to dataset files:", path)
path = "/home/dvipin2/.cache/kagglehub/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/versions/9/"

100%|█████████████████████████████████████████████████████████████████████████████████| 21.0G/21.0G [30:51<00:00, 12.2MB/s]


Extracting files...
Path to dataset files: /home/dvipin2/.cache/kagglehub/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/versions/9


In [4]:
! mkdir -p data
! cp /home/dvipin2/.cache/kagglehub/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/versions/9/amazon_reviews_us_Camera_v1_00.tsv data/

In [ ]:
import pandas as pd 
import os

path = "/home/dvipin2/.cache/kagglehub/datasets/cynthiarempel/amazon-us-customer-reviews-dataset/versions/9/"

# 2. Pick a single category file to read (e.g., Camera or Electronics)
file_name = "amazon_reviews_us_Camera_v1_00.tsv" 
full_file_path = os.path.join(path, file_name)

file_name2 = "amazon_reviews_us_Electronics_v1_00.tsv"
full_file_path2 = os.path.join(path, file_name2)

# 3. Read only the first 30,000 rows
df_camera = pd.read_csv(full_file_path, sep="\t", nrows=15000, on_bad_lines="skip")
df_electronics = pd.read_csv(full_file_path2, sep="\t", nrows=15000, on_bad_lines="skip")

df = pd.concat([df_camera, df_electronics])

# 4. Standardize columns to match our project schema
df_clean = df[[
    "product_id", 
    "customer_id", 
    "product_title", 
    "review_body", 
    "star_rating", 
    "product_category"
]].dropna()

In [21]:
df_clean.to_csv("data/sample_amazon_reviews.csv", index=False)
print(f"Sample created successfully with {len(df_clean)} rows!")

Sample created successfully with 29995 rows!


In [22]:
df_clean.head()

,product_id,customer_id,product_title,review_body,star_rating,product_category
0,B00I01JQJM,2975964,GoPro Rechargeable Battery 2.0 (HERO3/HERO3+ o...,ok,5,Camera
1,B00TCO0ZAA,23526356,Professional 58mm Center Pinch Lens Cap for CA...,"Perfect, even sturdier than the original!",5,Camera
2,B00B7733E0,52764145,Spy Tec Z12 Motion Activated Intelligent Secur...,"If the words, &#34;Cheap Chinese Junk&#34; com...",2,Camera
3,B006ZN4U34,47348933,"Celestron UpClose G2 10x25 Monocular, Black (7...",Exactly what I wanted and expected. Perfect fo...,5,Camera
4,B00HUEBGMU,33680700,Vidpro XM-L Wired Lavalier microphone - 20' Au...,I will look past the fact that they tricked me...,5,Camera


In [10]:
import math
import re
from collections import defaultdict

class InvertedIndex:
    def __init__(self):
        self.index = defaultdict(list)
        self.doc_lengths = {}
        self.avg_doc_len = 0
        self.corpus = {}

    def _tokenize(self, text):
        if not isinstance(text, str):
            return []
        text = re.sub(r'[^\w\s]', '', text.lower())
        return text.split()

    def build_index(self, df, id_col='product_id', text_col='review_body'):
        total_len = 0
        for idx, row in df.iterrows():
            doc_id = row[id_col]
            tokens = self._tokenize(str(row[text_col]))
            
            self.corpus[doc_id] = row.to_dict()
            self.doc_lengths[doc_id] = len(tokens)
            total_len += len(tokens)

            # Store term frequencies
            tf_counts = defaultdict(int)
            for token in tokens:
                tf_counts[token] += 1
            
            for token, count in tf_counts.items():
                self.index[token].append((doc_id, count))
                
        self.avg_doc_len = total_len / max(len(df), 1)

    def search_bm25(self, query, k1=1.5, b=0.75, top_k=10):
        query_tokens = self._tokenize(query)
        scores = defaultdict(float)
        N = len(self.doc_lengths)

        for token in query_tokens:
            if token not in self.index:
                continue
            postings = self.index[token]
            df = len(postings)
            idf = math.log((N - df + 0.5) / (df + 0.5) + 1)

            for doc_id, tf in postings:
                doc_len = self.doc_lengths[doc_id]
                denom = tf + k1 * (1 - b + b * (doc_len / self.avg_doc_len))
                scores[doc_id] += idf * ((tf * (k1 + 1)) / denom)

        sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
        return [(self.corpus[doc_id], score) for doc_id, score in sorted_docs]

In [12]:
! pip install networkx

  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)


In [13]:
import networkx as nx
from collections import defaultdict

class GraphRanker:
    def __init__(self):
        self.graph = nx.DiGraph()

    def build_co_review_graph(self, df):
        """Creates directed product edges based on shared user reviews"""
        user_to_products = defaultdict(set)
        for _, row in df.iterrows():
            user_to_products[row['customer_id']].add(row['product_id'])

        # Edge from Product A -> Product B if same user reviewed both
        for products in user_to_products.values():
            prod_list = list(products)
            for i in range(len(prod_list)):
                for j in range(i + 1, len(prod_list)):
                    self.graph.add_edge(prod_list[i], prod_list[j])
                    self.graph.add_edge(prod_list[j], prod_list[i])

    def compute_pagerank(self, alpha=0.85):
        return nx.pagerank(self.graph, alpha=alpha) if len(self.graph) > 0 else {}

    def compute_hits(self, max_iter=100):
        if len(self.graph) == 0:
            return {}, {}
        hubs, authorities = nx.hits(self.graph, max_iter=max_iter)
        return hubs, authorities

In [16]:
!pip install streamlit scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 10.8 MB/s  0:00:03m0:00:0100:01


In [17]:
import streamlit as st
import pandas as pd
#from utils.indexer import InvertedIndex
#from utils.ranker import GraphRanker

st.set_page_config(page_title="IR Engine & Graph Ranking", layout="wide")
st.title("Search Engine & Graph Ranking")

@st.cache_resource
def load_and_index():
    df = pd.read_csv("data/sample_amazon_reviews.csv").dropna()
    
    idx = InvertedIndex()
    idx.build_index(df)
    
    ranker = GraphRanker()
    ranker.build_co_review_graph(df)
    pr_scores = ranker.compute_pagerank()
    
    return idx, pr_scores

idx, pr_scores = load_and_index()

query = st.text_input("Enter product search query:", "camera lens")

if query:
    st.subheader("BM25 Standard Search Results")
    results = idx.search_bm25(query, top_k=5)
    
    for item, score in results:
        pid = item['product_id']
        pr = pr_scores.get(pid, 0.0)
        st.write(f"**{item['product_title']}** | ID: `{pid}` | BM25 Score: {score:.4f} | PageRank: {pr:.6f}")

2026-08-17 02:46:45.794 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 02:46:45.802 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 02:46:45.805 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 02:46:45.808 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 02:46:45.811 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 02:46:56.642 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 02:46:56.645 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-17 02:46:56.657 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar